# Test the fine-tuned Gemma3 1B model

Loads the model produced by `finetune.py` (saved to `/workspace/finetuned.keras`) and checks the instruction-following behavior with `generate()`, using the same prompt format the model was fine-tuned on:

```
[instruction]
{instruction}[end]
[response]
```

Note: running finetune.py on A100 SXM took only 6 minutes.

In [1]:
import os

os.environ["KERAS_BACKEND"] = "jax"
#os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

import keras
import keras_hub

## Load the fine-tuned model

Primary path: load the full native `.keras` model. `keras_hub` must be imported first so its custom layers are registered for deserialization.

In [3]:
gemma_lm = keras.models.load_model("finetuned.keras")

/Users/rjljr/devel/deep-learning-with-python-notebooks/.venv/lib/python3.12/site-packages/keras/src/saving/serialization_lib.py:749: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)
/Users/rjljr/devel/deep-learning-with-python-notebooks/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 210 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
normalizer.cc(51) LOG(INFO) precompiled_charsmap is empty. use identity normalization.


Fallback (if the `.keras` file is unavailable): rebuild the preset, re-enable LoRA, and load the backup weights.

```python
gemma_lm = keras_hub.models.CausalLM.from_preset("gemma3_1b", dtype="float32")
gemma_lm.backbone.enable_lora(rank=8)
gemma_lm.load_weights("/workspace/finetuned.weights.h5")
```

## Generate responses

Same instruction prompts used to validate the fine-tune in the chapter notebook.

In [4]:
gemma_lm.generate(
    "[instruction]\nHow can I make brownies?[end]\n"
    "[response]\n",
    max_length=512,
)

'[instruction]\nHow can I make brownies?[end]\n[response]\nBrownies are a type of cake that is made with a mixture of flour, sugar, and eggs. They are typically baked in a pan and then topped with chocolate or other toppings. You can make brownies with different flavors and textures, such as chocolate, vanilla, and caramel. You can also add other ingredients like nuts, chocolate chips, or raisins to the mixture to make them more flavorful and interesting.[end]'

In [6]:
print(gemma_lm.generate(
    "[instruction]\nWhat is a proper noun?[end]\n"
    "[response]\n",
    max_length=512,
))

[instruction]
What is a proper noun?[end]
[response]
A proper noun is a word that refers to a specific person, place, or thing. Proper nouns are usually capitalized and are used to identify specific individuals, places, or things. Proper nouns are often used in formal writing and are often used in titles, such as "The White House" or "The Eiffel Tower." Proper nouns are also used in titles of books, movies, and other works of literature.[end]


In [ ]:
gemma_lm.generate(
    "[instruction]\nWho is the 542nd president of the United States?[end]\n"
    "[response]\n",
    max_length=512,
)